# Titanic Survival Prediction
This notebook demonstrates a complete machine learning workflow to predict passenger survival on the Titanic using Logistic Regression.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.linear_model import LogisticRegression

## 1. Data Loading and Initial Exploration
First, we load the dataset and take a look at a few sample rows to understand the features we are working with.

In [2]:
df = pd.read_csv("titanic_data_updated.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
639,640,no,third,"Thorneycroft, Mr. Percival",male,NaN,1,0,376564,16.1000,NaN,S
365,366,no,third,"Adahl, Mr. Mauritz Nils Martin",male,30.0,0,0,C 7076,7.2500,NaN,S
266,267,no,third,"Panula, Mr. Ernesti Arvid",male,16.0,4,1,3101295,39.6875,NaN,S
659,660,no,first,"Newell, Mr. Arthur Webster",male,58.0,0,2,35273,113.2750,D48,C
359,360,yes,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,NaN,Q


In [5]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [3]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


## 2. Feature Engineering
We create new features like `Family_Size` and extract the `Deck` from the `Cabin` column to provide more meaningful information to the model.

In [6]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
17,18,yes,second,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,Missing,S,1,M
525,526,no,third,"Farrell, Mr. James",male,40.5,0,0,367232,7.7500,Missing,Q,1,M
721,722,no,third,"Jensen, Mr. Svend Lauritz",male,17.0,1,0,350048,7.0542,Missing,S,2,M
561,562,no,third,"Sivic, Mr. Husein",male,40.0,0,0,349251,7.8958,Missing,S,1,M
454,455,no,third,"Peduzzi, Mr. Joseph",male,NaN,0,0,A/5 2817,8.0500,Missing,S,1,M


In [7]:
df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [8]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

In [9]:
# Split data: 80% for training the model and 20% for testing its accuracy
# 'stratify=y' ensures both sets have a similar percentage of survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Outlier Handling and Data Cleaning
We use Z-scores to remove extreme outliers in `Age` and 'clip' the `Fare` column to prevent extreme values from distorting our model's learning.

In [10]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [12]:
mean_age=X_train['Age'].mean()
std_age=X_train['Age'].mean()

X_train['Z_score']=(X_train['Age']-mean_age)/std_age

musk=(abs(X_train['Z_score'])<=3)

X_train=X_train[musk]
y_train=y_train[musk]

In [17]:
#fare

q1=X_train['Fare'].quantile(0.25)
q3=X_train['Fare'].quantile(0.75)

iqr=q3-q1

lb=max(0,q1-1.5*iqr)
ub=q3+1.5*iqr

X_train['Fare']=X_train['Fare'].clip(lb,ub)


/tmp/ipykernel_1077/2404049884.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare']=X_train['Fare'].clip(lb,ub)


## 5. Building Preprocessing Pipelines
We create different pipelines for numerical and categorical data. This handles missing values (imputation) and scales the data so all features are on a similar range.

In [18]:
from warnings import simplefilter
p1=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('std_scaler',StandardScaler())
    ]
)

p2=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [21]:
p1

Pipeline(steps=[('imputer', SimpleImputer()), ('std_scaler', StandardScaler())])

In [22]:
p2

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', MinMaxScaler())])

In [23]:
categories = [['third','second','first']]

In [25]:
p3=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4=Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('std_scaler',
                                                  StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

## 6. Label Encoding
We convert our target variable (`yes`/`no`) into numerical values (1/0) because machine learning models require numeric inputs.

In [30]:
le = LabelEncoder()

y_train=le.fit_transform(y_train)
y_test = le.transform(y_test)

## 7. Model Training
We combine the preprocessing and the Logistic Regression model into a single `lr_model` pipeline and fit it to our training data.

In [31]:
lr_model=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',LogisticRegression(class_weight='balanced',max_iter=1000))
    ]
)

lr_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline_1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('std_scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_Size']),
                                                 ('pipeline_3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(str...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [32]:
lr_model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline_1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('std_scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_Size']),
                                                 ('pipeline_3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(str...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [33]:
print(lr_model['model'].coef_)
print(lr_model['model'].intercept_)
print(lr_model['model'].classes_)

[[-0.55677758  0.5311947  -0.97181134 -0.30077564 -0.31934779 -2.40470843
  -0.22812707 -0.34527844  0.36404756  0.65101841  0.44391813 -0.59776867
  -0.71236308 -0.34307425  1.49693631]]
[1.53243729]
[0 1]


In [34]:
# Use the trained model to predict if passengers in the test set survived
y_pred = lr_model.predict(X_test)

# Get the raw probability scores (e.g., 0.85 chance of death, 0.15 chance of survival)
lr_model.predict_proba(X_test)

array([[0.85500733, 0.14499267],
       [0.92174397, 0.07825603],
       [0.77369249, 0.22630751],
       [0.92448153, 0.07551847],
       [0.37791038, 0.62208962],
       [0.43632533, 0.56367467],
       [0.24788908, 0.75211092],
       [0.57571888, 0.42428112],
       [0.47321943, 0.52678057],
       [0.77864041, 0.22135959],
       [0.86133595, 0.13866405],
       [0.81957353, 0.18042647],
       [0.33870905, 0.66129095],
       [0.70642528, 0.29357472],
       [0.15667685, 0.84332315],
       [0.83774391, 0.16225609],
       [0.46525277, 0.53474723],
       [0.86327003, 0.13672997],
       [0.79527705, 0.20472295],
       [0.23382374, 0.76617626],
       [0.86327003, 0.13672997],
       [0.20023945, 0.79976055],
       [0.86414097, 0.13585903],
       [0.45576117, 0.54423883],
       [0.8597215 , 0.1402785 ],
       [0.0278666 , 0.9721334 ],
       [0.66153659, 0.33846341],
       [0.6184413 , 0.3815587 ],
       [0.81659893, 0.18340107],
       [0.81375603, 0.18624397],
       [0.

## 8. Evaluation
Finally, we evaluate the model using accuracy, precision, and recall to see how well it performs on the test set.

In [35]:
from sklearn.metrics import accuracy_score,precision_score,recall_score

In [36]:
accuracy = accuracy_score(y_test,y_pred)
print(accuracy)
precision = precision_score(y_test,y_pred)
print(precision)
recall = recall_score(y_test,y_pred)
print(recall)

0.7653631284916201
0.6753246753246753
0.7536231884057971
